# Install Packages

In [1]:
# pip install git+https://github.com/dnth/rag-datakit.git
# !uv pip install tiktoken

In [2]:
#!uv pip install ipywidgets
#!uv pip install python-dotenv


# Load ENV

In [1]:
import os
from huggingface_hub import login
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get token from environment
token = os.getenv("HF_TOKEN")
login(token=token)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


# Load SSF Data

In [2]:
from datasets import load_dataset

dataset = load_dataset("dnth/ssf-dataset")
dataset

DatasetDict({
    train: Dataset({
        features: ['Sector', 'Track', 'Job Role', 'Job Role Description', 'Performance Expectation'],
        num_rows: 1885
    })
})

In [3]:
dataset["train"][0]

{'Sector': 'Accountancy',
 'Track': 'Assurance',
 'Job Role': 'Audit Associate / Audit Assistant Associate',
 'Job Role Description': 'The Audit Associate/Audit Assistant Associate undertakes specific stages of audit work under supervision. He/She begins to appreciate the underlying principles behind the tasks assigned to him as part of the audit plan. He is also able to make adjustments to the application of skills to improve the work tasks or solve non-complex issues. The Audit Associate/Audit Assistant Associate operates in a structured work environment. He is able to build relationships, work in a team and identify ethical issues with reference to the code of professional conduct and ethics. He is able to select and apply from a range of known solutions to familiar problems and takes responsibility for his own learning and performance. He is a trustworthy and meticulous individual.',
 'Performance Expectation': 'In accordance with: Singapore Standards on Auditing, Ethics Pronouncem

# SSF Job Description Token Analysis

In [4]:
import tiktoken
from datasets import load_dataset
import numpy as np
import math

# Load the OpenAI API key from environment

text_column = 'Job Role Description'

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Get the Job Role Description column
job_descriptions = dataset['train'][text_column]

# Calculate token lengths for all job descriptions
token_lengths = [num_tokens_from_string(description, "cl100k_base") for description in job_descriptions]

# Calculate the average, max, min, and other statistics
average_token_length = np.mean(token_lengths)
max_token_length = np.max(token_lengths)
min_token_length = np.min(token_lengths)
std_dev_token_length = np.std(token_lengths)  # Standard deviation

# Calculate percentiles
p25 = np.percentile(token_lengths, 25)  # 25th percentile
p50 = np.percentile(token_lengths, 50)  # 50th percentile (median)
p75 = np.percentile(token_lengths, 75)  # 75th percentile

# Calculate IQR (Interquartile Range)
IQR = p75 - p25
lower_bound = p25 - 1.5 * IQR
upper_bound = p75 + 1.5 * IQR

# Detect outliers
outliers = [length for length in token_lengths if length < lower_bound or length > upper_bound]

# Print the exact average token length
print(f"Exact average token length: {average_token_length}")

# Round up the average token length to the nearest integer
token_avg_length_rounded = math.ceil(average_token_length)

# Print the rounded-up average token length
print(f"Rounded up average token length: ~{token_avg_length_rounded}")

# Print the max and min token lengths
print(f"Maximum token length: {max_token_length}")
print(f"Minimum token length: {min_token_length}")

# Print the standard deviation and variance
print(f"Standard deviation: {std_dev_token_length}")

# Print percentiles
print(f"25th Percentile: {p25}")
print(f"50th Percentile (Median): {p50}")
print(f"75th Percentile: {p75}")

# Print outliers
print(f"Number of outliers: {len(outliers)}")



Exact average token length: 161.60371352785145
Rounded up average token length: ~162
Maximum token length: 393
Minimum token length: 52
Standard deviation: 49.290052444984674
25th Percentile: 124.0
50th Percentile (Median): 155.0
75th Percentile: 195.0
Number of outliers: 14


# Synthetic Data Generation Setup

In [5]:
import os
from distilabel.models import OpenAILLM, TransformersLLM

# llm = TransformersLLM(
#     model="Qwen/Qwen3-4B-Instruct-2507",
#     device_map="auto",
#     torch_dtype="float16",
# )

llm = OpenAILLM(
    model="gpt-4o-mini",
    # model="gpt-5-mini-2025-08-07",
    api_key=os.getenv("OPENAI_API_KEY"),
)


In [6]:
context = """
You are an HR assistant tasked with generating realistic job descriptions based on a Singapore SkillsFuture Framework input. 
For each job, you will create **one positive description** and **five negative description**.

### Input:
A job description containing:
- Job title (e.g., Audit Associate)
- Role responsibilities and duties
- Work environment and supervision structure
- Required skills and attributes
- Professional conduct expectations

### Output Instructions:

#### 1. Positive Description
- Always start the Positive job desccription with "The [Job Role]"
- Capture the essence of the original role using different words
- Keep the same seniority level and core responsibilities
- Use varied terminology naturally
- Include specific responsibilities, skills, and requirements
- Read as if a different organization is posting a similar role

#### 2. Negative Description
- Always start the Negative job descriptions with "The [Job Role]" 
- Don't label the negative type
- Include some similar keywords but **change the intent, context, or responsibilities**
- create five different negative descriptions using the strategies below

**Negative Strategies**:

1. **Easy Negative - Different Function, Same Industry**
   - Change the core function but keep the same industry
   - Use completely different skills
   - Maintain professional context
   - Example: Audit Associate → Tax Associate

2. **Medium Negative - Same Industry, Different Seniority**
   - Change responsibility level (Junior ↔ Senior)
   - Alter supervision structure
   - Modify years of experience or decision-making authority
   - Example: Audit Associate → Senior Audit Manager

3. **Hard Negative - Same Skills, Different Domain**
   - Transfer core skills to a different industry
   - Maintain similar analytical/technical requirements
   - Change regulatory environment or business context
   - Example: Audit Associate → Compliance Associate (Banking)

4. **Hard Negative - Geographic/Regulatory Variation**
   - Same role but different regulatory or geographic context
   - Vary market maturity and business practices
   - Include cross-border or international elements

5. **Very Hard Negative - Hybrid Role Confusion**
   - Combine responsibilities from multiple distinct roles
   - Create plausible but incorrect role combinations
   - Mix strategic and tactical responsibilities inappropriately
   - Include overlapping but different skill requirements

### Output Format for Positive Description:
The [Job Role] ...

### Output Format for the 5 Negative Descriptions:
The [Job Role] ... 

The [Job Role] ...

The [Job Role] ...

The [Job Role] ...

The [Job Role] ...   
"""



In [9]:
from distilabel.pipeline import Pipeline
from distilabel.steps import LoadDataFromHub
from distilabel.steps.tasks import GenerateSentencePair

with Pipeline(name="generate") as pipeline:
    load_dataset = LoadDataFromHub(
        #num_examples=100,  # Limit to 10 examples for demo - increase for production datasets
        use_cache=False,  # Disable caching to ensure fresh data generation each run
        output_mappings={"Job Role Description": "anchor"},  # Map original column to 'anchor' for triplet generation
    )
    generate_retrieval_pairs_easy = GenerateSentencePair(
        name="easy_triplets_paraphrase",
        triplet=True,  # Generate anchor-positive-negative triplets for embedding training
        hard_negative=False,  # Use easier negatives rather than hard negatives
        action="paraphrase",  # Focus on paraphrasing for positive examples
        llm=llm,  # Use the LLM configured above (local Qwen or OpenAI)
        # input_batch_size=10,  # Process 10 examples at once for efficiency
        input_batch_size=10,  # Process 10 examples at once for efficiency
        context=context,  # Provide the context instructions for generation quality
    )
    generate_retrieval_pairs_hard = GenerateSentencePair(
        name="hard_triplets_paraphrase",
        triplet=True,  
        hard_negative=True,  
        action="paraphrase",  
        llm=llm,  
        # input_batch_size=10,  
        input_batch_size=10,  
        context=context,  
    )

    load_dataset.connect(generate_retrieval_pairs_easy, generate_retrieval_pairs_hard)

In [10]:
output_avg_token_length = token_avg_length_rounded*2
# output_max_token_length = max_token_length*2
output_max_token_length = max_token_length*6

print("Output Token Length:", output_avg_token_length)
print("Output Max Token Length:", output_max_token_length)

Output Token Length: 324
Output Max Token Length: 2358


In [11]:
distiset = pipeline.run(
    use_cache=False,
    parameters={
        load_dataset.name: {
            "repo_id": "dnth/ssf-dataset",
            "split": "train",
        },
        "easy_triplets_paraphrase": {
            "llm": {"generation_kwargs": {"temperature": 0.6, "max_new_tokens": output_max_token_length}}
             #"llm": {"generation_kwargs": {"temperature": 0.6}}
        },
        "hard_triplets_paraphrase": {
            "llm": {"generation_kwargs": {"temperature": 0.6, "max_new_tokens": output_max_token_length}}
             #"llm": {"generation_kwargs": {"temperature": 0.6}}
        },
    }
)

[09/11/25 10:36:39] INFO     ['distilabel.pipeline'] 📝 Pipeline data will be written to               ]8;id=196101;file:///home/frank123/GitHub/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=574731;file:///home/frank123/GitHub/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1015\1015]8;;\
                             '/home/frank123/.cache/distilabel/pipelines/generate/5452575778c9f9fcb171             
                             f296279096b5c1119436/executions/ed2cb6a02e89ca97af074b5e0060c2873ffe9855/             
                             data/steps_outputs'                                                                   

                    INFO     ['distilabel.pipeline'] ⌛ The steps of the pipeline will be loaded in    ]8;id=938370;file:///home/frank123/GitHub/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=64160;file:///home/frank123/GitHub/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1046\1046]8;;\
                             stages:                                                                               
                              * Legend: 🚰 GeneratorStep 🌐 GlobalStep 🔄 Step                                     
                              * Stage 0:                                                                           
                                - 🚰 'load_data_from_hub_0'                                                        
                                - 🔄 'easy_triplets_paraphrase'                                                    
                                - 🔄 'hard_triplets_paraphrase'                                                    

                    INFO     ['distilabel.pipeline'] ⏳ Waiting for all the steps of stage 0 to        ]8;id=625024;file:///home/frank123/GitHub/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=928308;file:///home/frank123/GitHub/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1382\1382]8;;\
                             load...                                                                               

[09/11/25 10:36:42] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 2/3                 ]8;id=467127;file:///home/frank123/GitHub/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=53350;file:///home/frank123/GitHub/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_hub_0' replicas: 0/1                                               
                              * 'easy_triplets_paraphrase' replicas: 1/1                                           
                              * 'hard_triplets_paraphrase' replicas: 1/1                                           

[09/11/25 10:36:49] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 3/3                 ]8;id=870116;file:///home/frank123/GitHub/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=770397;file:///home/frank123/GitHub/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_hub_0' replicas: 1/1                                               
                              * 'easy_triplets_paraphrase' replicas: 1/1                                           
                              * 'hard_triplets_paraphrase' replicas: 1/1                                           

...

                    INFO     ['distilabel.step.hard_triplets_paraphrase'] 🏁 Finished running   ]8;id=205072;file:///home/frank123/GitHub/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=625914;file:///home/frank123/GitHub/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/step_wrapper.py#129\129]8;;\
                             step 'hard_triplets_paraphrase' (replica ID: 0)                                       

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [12]:
distiset

Distiset({
    easy_triplets_paraphrase: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 1885
        })
    })
    hard_triplets_paraphrase: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 1885
        })
    })
})

In [13]:
distiset["hard_triplets_paraphrase"]["train"][-1]

{'Sector': 'Workplace Safety and Health',
 'Track': 'System Audit',
 'Job Role': 'Workplace Safety and Health Auditor',
 'anchor': 'The WSH Auditor is responsible for preparing audit plans, conducting audits and interviews and submitting audit report. He/she is responsible for evaluating an organisations WSH management system, identify areas for improvement, make the relevant recommendations and monitor the progress of improvement. In addition, he is expected to conduct physical inspection of workplace to collect and verify information in accordance to the audit plan. The WSH Auditor is analytical, resourceful, collaborative and has good teamwork.',
 'Performance Expectation': 'In accordance with: Workplace Safety and Health Act',
 'positive': "The WSH Auditor is tasked with developing comprehensive audit strategies, executing audits and interviews, and delivering detailed audit reports. This role involves assessing an organization's Workplace Safety and Health (WSH) management framewo

In [14]:
hard_triplets_semantic_df = distiset["hard_triplets_paraphrase"]["train"].to_pandas()
# hard_triplets_semantic_df = distiset["easy_triplets_paraphrase"]["train"].to_pandas()
hard_triplets_semantic_df

,Sector,Track,Job Role,anchor,Performance Expectation,positive,negative,distilabel_metadata,model_name
0,Accountancy,Assurance,Audit Associate / Audit Assistant Associate,The Audit Associate/Audit Assistant Associate ...,In accordance with: Singapore Standards on Aud...,The Audit Associate is responsible for executi...,The Tax Associate manages various aspects of t...,{'raw_input_hard_triplets_paraphrase': [{'cont...,gpt-4o-mini
1,Accountancy,Assurance,Audit Manager,The Audit Senior Manager/Audit Manager manages...,In accordance with: Singapore Standards on Aud...,The Audit Senior Manager oversees a diverse po...,The Audit Senior Manager coordinates a series ...,{'raw_input_hard_triplets_paraphrase': [{'cont...,gpt-4o-mini
2,Accountancy,Assurance,Audit Partner / Audit Director,The Audit Partner/Audit Director is a transfor...,In accordance with: Singapore Standards on Aud...,The Audit Director is a visionary leader who g...,The Audit Partner is responsible for managing ...,{'raw_input_hard_triplets_paraphrase': [{'cont...,gpt-4o-mini
3,Accountancy,Assurance,Audit Senior,The Audit Senior is expected to team lead vari...,In accordance with: Singapore Standards on Aud...,The Audit Senior is responsible for leading di...,The Audit Senior is tasked with overseeing var...,{'raw_input_hard_triplets_paraphrase': [{'cont...,gpt-4o-mini
4,Accountancy,Business Valuation,Business Valuation Associate / Business Valuat...,The Business Valuation Associate/Business Valu...,In accordance with the International Valuation...,The Business Valuation Associate is tasked wit...,The Business Valuation Associate is responsibl...,{'raw_input_hard_triplets_paraphrase': [{'cont...,gpt-4o-mini
...,...,...,...,...,...,...,...,...,...
1880,Workplace Safety and Health,Operational Control,Workplace Safety and Health Manager,The WSH Manager is responsible for reviewing W...,In accordance with: Workplace Safety and Healt...,The WSH Manager is tasked with evaluating and ...,The WSH Manager is focused on developing marke...,{'raw_input_hard_triplets_paraphrase': [{'cont...,gpt-4o-mini
1881,Workplace Safety and Health,Operational Control,Workplace Safety and Health Officer,The WSH Officer is responsible for developing ...,In accordance with: Workplace Safety and Healt...,The WSH Officer is tasked with creating and ov...,The WSH Officer is responsible for managing th...,{'raw_input_hard_triplets_paraphrase': [{'cont...,gpt-4o-mini
1882,Workplace Safety and Health,Operational Control,Workplace Safety and Health Supervisor,The Workplace Safety and Health (WSH) Supervis...,In accordance with: Workplace Safety and Healt...,The Workplace Safety and Health (WSH) Supervis...,The Workplace Safety and Health (WSH) Coordina...,{'raw_input_hard_triplets_paraphrase': [{'cont...,gpt-4o-mini
1883,Workplace Safety and Health,System Audit,Lead Workplace Safety and Health Auditor,The Lead Workplace Safety and Health (WSH) Aud...,In accordance with: Workplace Safety and Healt...,The Lead Workplace Safety and Health (WSH) Aud...,The Lead Workplace Safety and Health (WSH) Con...,{'raw_input_hard_triplets_paraphrase': [{'cont...,gpt-4o-mini


In [15]:
#distiset.push_to_hub("frankwong2001/ssf-dataset-synthetic_test_2")
distiset.push_to_hub("frankwong2001/ssf-dataset_Full_synthetic_v3")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  10%|#         |  528kB / 5.10MB            

...

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   4%|3         |  289kB / 7.47MB            

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]